In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from articulate import articulate
from omegaconf import OmegaConf
from dotenv import load_dotenv
from articulate_anything.utils.viz import (
    show_video, 
    display_code, 
    show_videos, 
    display_codes,
    show_images,
)
from articulate_anything.utils.utils import load_config, join_path
from articulate_anything.utils.cotracker_utils import make_cotracker
from PIL import Image
import json

In [ ]:
import os
os.chdir("..")

In [ ]:
API_KEY = "YOUR-ACTUAL-API-KEY"
## we have our API key stored in a .env file
load_dotenv()
API_KEY = os.environ.get('API_KEY')

In [ ]:
modality = "partnet"
prompt = "35059"
joint_id = "joint_0"
out_dir = "results"

In [ ]:
cfg = load_config()
cfg.prompt = prompt
cfg.additional_prompt = joint_id
cfg.modality = modality
cfg.out_dir = out_dir
cfg.api_key = API_KEY

cfg.model_name = "claude-3-5-sonnet-20241022"


cfg.cotracker.grid_size = 30
cfg.cotracker.mode = "offline"


cfg.actor_critic.max_iter = 2

cfg.joint_critic.examples_dir = "datasets/multi_modal_incontext_examples/joint_critic/in_context_examples_datasets" ## Put your examples here

cfg.simulator.ray_tracing = False

In [ ]:
cfg.cotracker

In [ ]:
use_cotracker = True
cfg.joint_actor.use_cotracker = use_cotracker
cfg.joint_critic.use_cotracker = use_cotracker
cfg.cotracker.checkpoint_path = "../co-tracker/checkpoints/cotracker2.pth" ## set it to your correct path

In [ ]:
steps = articulate(cfg)

## Link Placement

In [ ]:
link_art = steps["Link Articulation"]
link_actors = link_art["Link actor"]
link_critics = link_art["Link critic"]
assert len(link_actors) == len(link_critics)
print(f"Link placement runs for {len(link_actors)} iteration(s)")

In [ ]:
link_feedbacks = [json.dumps(link_critic.load_prediction(),
                             indent=4) for link_critic in link_critics]

link_codes = [link_actor.load_prediction() for link_actor in link_actors]

link_preds = [link_actor.load_predicted_rendering() for link_actor in link_actors]

Here's the code to place the links in the 3D space

In [ ]:
show_images(link_preds)
display_codes(link_codes)
display_codes(link_feedbacks)

Let's see what our own critic has to say about the link placement

## Joint Prediction

In [ ]:
joint_art = steps["Joint Articulation"]
joint_actors = joint_art["Joint actor"]
joint_critics = joint_art["Joint critic"]


assert len(joint_actors) == len(joint_critics)
print(f"Joint prediction runs for {len(joint_actors)} iteration(s)")

In [ ]:
joint_codes = [joint_actor.load_prediction() for joint_actor in joint_actors]
joint_preds = [joint_actor.load_predicted_rendering() for joint_actor in joint_actors]
joint_feedbacks = [json.dumps(joint_critic.load_prediction(),
                             indent=4) for joint_critic in joint_critics]

In [ ]:
show_videos(joint_preds, width=512, height=512)
display_codes(joint_codes)
display_codes(joint_feedbacks)